# 09 — Vendor HDS-Style Feature Importance on L5B15

Mirrors Pega's *Historical Data Set Analysis* notebook (`pdstools`) on the Transavia L5B15 dataset, and compares the resulting feature importances against this thesis's SHAP / LIME / Decision-Tree rankings from notebook 06.

## Important caveat

Pega's HDS notebook expects a **captured `Decision_Outcome`** (Accepted/Rejected). Our `data/decisions/` export contains the model **inputs and the propensity score** Pega's NaiveBayes produced, but **no row-level customer outcome**. Outcomes only exist in aggregated form inside the ADM snapshots (`pyPositives` / `pyNegatives` per model and per predictor-bin).

Following the same logic as notebook 05's CatBoost surrogate, we therefore **binarize Pega's propensity score** (top decile → `Accepted`, rest → `Rejected`) to get a row-level target. This means the XGBoost classifier is learning *what Pega's model rewards*, not the true business outcome. For the purpose of comparing **which features matter**, that's the same question the SHAP / LIME / DT analyses ask, so the comparison is meaningful.

## What we run

1. Load L5B15 parquet (notebook 02 output).
2. Rename columns to the underscore-prefix convention Pega's HDS uses (`Customer_*`, `Context_*`, `IH_*`, `Param_*`, plus Transavia-specific `Booking_*`, `Precompute_*`).
3. Build the field/category dictionary.
4. Binarize `propensity` → `Decision_Outcome`.
5. Run their `data_prep` (label-encode categoricals) → `XGBClassifier` → `plot_feature_imp` (importance bar chart with correlation grouping).
6. Compare top-N XGBoost importances vs. the thesis SHAP / LIME / DT rankings (Spearman ρ, Jaccard@k, side-by-side chart).


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

from my_project.metrics import stability_spearman, jaccard_at_k

REPO = Path('..').resolve()
# Load the up-to-date all-luggage parquet (99,596 rows from all 3 JSON ingests);
# l5b15_email_outbound.parquet is stale at 2,268 rows.
PARQUET = REPO / 'data' / 'processed' / 'luggage_email_outbound.parquet'
ARTIFACTS = REPO / 'data' / 'artifacts' / 'l5b15'
OUT_DIR = REPO / 'data' / 'artifacts' / 'l5b15' / 'vendor_hds'
OUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANT = 'L5B15'
PRODUCTION_TECHNIQUE = '0.0'    # matches notebook 08
RANDOM_STATE = 42
TOP_DECILE = 0.90               # propensity quantile cutoff for Accepted


## 1. Load L5B15 and harmonise the schema

The parquet stores every predictor as a string (mirroring the raw Pega JSON). We:

- Drop ID columns and model-output columns (otherwise they leak the target).
- Cast columns to numeric where >80% of values parse as floats.
- Rename dot-paths to Pega-style underscore prefixes so the vendor's `Category = first underscore token` convention works.


In [ ]:
raw = pl.read_parquet(PARQUET)
print(f'Full luggage parquet: {raw.shape[0]:,} rows × {raw.shape[1]} cols')
print('modelTechnique counts:', raw["modelTechnique"].value_counts().to_dict())

df = raw.filter(
    (pl.col('pyName') == VARIANT) &
    (pl.col('modelTechnique') == PRODUCTION_TECHNIQUE)
)
print(f'After variant + production-technique filter ({VARIANT}, modelTechnique={PRODUCTION_TECHNIQUE}): {df.shape[0]:,} rows')

LEAK_COLS = {
    'pxInteractionID', 'pxSubjectID', 'modelExecutionID', 'modelVersion',
    'TreatmentID', 'propensity', 'modelPerformance', 'modelEvidence', 'modelTechnique',
}

propensity = df['propensity'].to_numpy()
df_features = df.drop([c for c in LEAK_COLS if c in df.columns])
print(f'After dropping leak cols: {df_features.shape[1]} feature cols')


In [ ]:
def rename_to_hds(col: str) -> str:
    """Map our dot-notation column names onto Pega's HDS underscore-prefix convention."""
    if col in {'pyName', 'pyChannel', 'pyDirection', 'pyGroup', 'pyIssue'}:
        return 'Context_' + col[2:]
    if col.startswith('Customer.'):
        return 'Customer_' + col[len('Customer.'):].replace('.', '_')
    if col.startswith('CustBookedFlight.'):
        return 'Booking_' + col[len('CustBookedFlight.'):].replace('.', '_')
    if col.startswith('CurrentContext.'):
        return 'Context_' + col[len('CurrentContext.'):].replace('.', '_')
    if col.startswith('PreComputes.'):
        return 'Precompute_' + col[len('PreComputes.'):].replace('.', '_')
    if col.startswith('IH.'):
        return 'IH_' + col[len('IH.'):].replace('.', '_')
    if col.startswith('param::Param.'):
        return 'Param_' + col[len('param::Param.'):].replace('.', '_')
    return 'Customer_' + col.replace('.', '_')

# Build a rename map, suffixing duplicates so rename() doesn't collide.
seen, rename_map = {}, {}
for c in df_features.columns:
    new = rename_to_hds(c)
    if new in seen:
        seen[new] += 1
        new = f'{new}__{seen[new]}'
    else:
        seen[new] = 0
    rename_map[c] = new
df_renamed = df_features.rename(rename_map)
print('Example renames:')
for k, v in list(rename_map.items())[:8]:
    print(f'  {k}  ->  {v}')


In [ ]:
# Try-cast each string column to Float64; keep numeric if ≥80% of non-null values parse.
numeric_cols = []
casts = []
for c in df_renamed.columns:
    dtype = df_renamed.schema[c]
    if dtype.is_numeric():
        numeric_cols.append(c); continue
    if dtype != pl.String:
        continue
    cast = df_renamed[c].cast(pl.Float64, strict=False)
    non_null = df_renamed[c].is_not_null().sum()
    parsed = cast.is_not_null().sum()
    if non_null > 0 and parsed / non_null >= 0.8:
        casts.append(cast.alias(c))
        numeric_cols.append(c)

if casts:
    df_renamed = df_renamed.with_columns(casts)
print(f'Numeric columns: {len(numeric_cols)} / {df_renamed.shape[1]}')


## 2. Build the data dictionary (vendor format)


In [ ]:
def category_of(field: str) -> str:
    if '_' in field:
        return field.split('_', 1)[0]
    return 'Internal'

dictionary = pl.DataFrame({
    'Field': df_renamed.columns,
    'Numeric': [df_renamed.schema[c].is_numeric() for c in df_renamed.columns],
}).with_columns(
    Category=pl.col('Field').map_elements(category_of, return_dtype=pl.Utf8)
).sort('Category', 'Field')

counts = (
    dictionary.group_by('Category', 'Numeric')
    .agg(Count=pl.len()).sort('Category')
)
px.bar(counts, y='Category', x='Count', color='Numeric', text='Count',
       orientation='h', title='Fields per category (L5B15 HDS-style)')


## 3. Binarize propensity → `Decision_Outcome`

Top-decile propensity is labelled `Accepted`; everything else `Rejected`. This gives ~10% positive class, similar to Pega's reference HDS dataset (~7% Accepted).

In [ ]:
threshold = float(np.quantile(propensity, TOP_DECILE))
outcome = np.where(propensity >= threshold, 'Accepted', 'Rejected')
df_renamed = df_renamed.with_columns(pl.Series('Decision_Outcome', outcome))
print(f'Threshold (q={TOP_DECILE}): {threshold:.6f}')
print(pd.Series(outcome).value_counts())


## 4. Vendor `data_prep` + `create_classifier`

Lifted from Pega's notebook, with cosmetic adjustments (drop the `Decision`/`Internal` categories from training, label-encode categoricals, 80/20 split).

In [ ]:
def data_prep(data: pl.DataFrame, data_dictionary: pl.DataFrame):
    excluded_categories = {'Internal', 'Decision'}
    categorical_fields = (
        data_dictionary.filter(~pl.col('Numeric'))
        .filter(~pl.col('Category').is_in(list(excluded_categories)))
        ['Field'].to_list()
    )
    numerical_fields = (
        data_dictionary.filter(pl.col('Numeric'))
        .filter(~pl.col('Category').is_in(list(excluded_categories)))
        ['Field'].to_list()
    )

    pdf = data.to_pandas()
    for col in categorical_fields:
        pdf[col] = pdf[col].fillna('missing').astype(str)
        le = LabelEncoder()
        pdf[col + '_encoded'] = le.fit_transform(pdf[col])

    le_target = LabelEncoder()
    pdf['target'] = le_target.fit_transform(pdf['Decision_Outcome'])
    target_mapping = dict(zip(le_target.classes_, range(len(le_target.classes_))))
    print(f'Target encoding: {target_mapping}')

    feature_cols = [c + '_encoded' for c in categorical_fields] + numerical_fields
    X = pdf[feature_cols]
    y = pdf['target']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f'Train: {X_train.shape}  Test: {X_test.shape}')
    return X_train, X_test, y_train, y_test, le_target, feature_cols


def create_classifier(X_train, X_test, y_train, y_test, target_encoder):
    model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(f'Model AUC: {roc_auc_score(y_test, y_proba):.4f}')
    print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))
    return model


X_train, X_test, y_train, y_test, target_encoder, feature_cols = data_prep(df_renamed, dictionary)
classifier = create_classifier(X_train, X_test, y_train, y_test, target_encoder)


## 5. Feature importance with correlation grouping

Exactly the vendor's `plot_feature_imp`: top-50 by XGBoost gain, with features ≥0.95 correlated grouped into the same `Group N` label, coloured by category.

In [ ]:
def plot_feature_imp(classifier, data: pd.DataFrame, feature_cols, top_n: int = 50):
    importances = classifier.feature_importances_
    fi = (
        pl.DataFrame({'Feature': feature_cols, 'Importance': importances.tolist()})
        .with_columns(
            Feature=pl.when(pl.col('Feature').str.ends_with('_encoded'))
            .then(pl.col('Feature').str.replace(r'_encoded$', ''))
            .otherwise(pl.col('Feature'))
        )
        .with_columns(
            Category=pl.col('Feature').map_elements(category_of, return_dtype=pl.Utf8)
        )
        .sort('Importance', descending=True)
    )
    top = fi.head(top_n)
    feature_order = top['Feature'].to_list()
    encoded_names = [
        f'{f}_encoded' if f'{f}_encoded' in feature_cols else f
        for f in feature_order
    ]

    # 0.95 correlation grouping
    groups = []
    for i in range(len(encoded_names)):
        for j in range(i + 1, len(encoded_names)):
            a, b = encoded_names[i], encoded_names[j]
            try:
                corr = data[[a, b]].corr().iloc[0, 1]
            except Exception:
                continue
            if pd.notna(corr) and abs(corr) >= 0.95:
                placed = False
                for k, grp in enumerate(groups):
                    if a in grp:
                        groups[k] = grp + (b,); placed = True; break
                    if b in grp:
                        groups[k] = grp + (a,); placed = True; break
                if not placed:
                    groups.append((a, b))
    group_label = {}
    for idx, grp in enumerate(groups, 1):
        for f in grp:
            group_label[f.replace('_encoded', '')] = f'Group {idx}'

    top = top.with_columns(
        pl.col('Feature').map_elements(lambda f: group_label.get(f, ''), return_dtype=pl.Utf8).alias('GroupLabel')
    )

    fig = px.bar(
        top, x='Importance', y='Feature', orientation='h',
        title=f'XGBoost Feature Importance — Top {top_n} (L5B15, HDS-style)',
        template='plotly_white', color='Category', text='GroupLabel',
        color_discrete_map={
            'Context': 'orange', 'IH': 'green', 'Customer': 'blue',
            'Param': 'lightblue', 'Booking': 'purple', 'Precompute': 'teal',
            'Internal': 'gray',
        },
    )
    fig.update_layout(
        xaxis_title='Importance', yaxis_title='Feature',
        yaxis=dict(categoryorder='array', categoryarray=feature_order,
                   autorange='reversed', dtick=1),
        height=900,
    )
    fig.update_traces(textposition='outside', textfont=dict(color='black', size=11))
    return fig, fi

fig_imp, full_importance = plot_feature_imp(classifier, X_train, feature_cols, top_n=50)
fig_imp.show()

# Save importance ranking for the comparison section
xgb_importance = {
    row['Feature']: row['Importance']
    for row in full_importance.iter_rows(named=True)
}
with open(OUT_DIR / 'vendor_xgb_importances.json', 'w') as f:
    json.dump(xgb_importance, f, indent=2)
print(f'Wrote {len(xgb_importance)} importances → {OUT_DIR / "vendor_xgb_importances.json"}')


## 6. Comparison vs. thesis SHAP / LIME / Decision Tree

The thesis fits a CatBoost surrogate on the **23 active Pega L5B15 predictors** only, then computes SHAP / LIME / DT importances on that 23-feature space. The vendor XGBoost above instead consumes *every* available HDS field.

For a fair comparison we restrict the XGBoost ranking to those 23 features. The original column names in the thesis use the dot-notation (`Customer.X`), so we apply the same `rename_to_hds` map.

**Metrics**

- **Spearman ρ** on the intersected feature ranking.
- **Jaccard@5 / @10** on the top-k feature sets.
- Side-by-side bar chart of the top-15 features per method.

In [ ]:
shap_imp = json.load(open(ARTIFACTS / 'shap_importances.json'))
lime_imp = json.load(open(ARTIFACTS / 'lime_importances.json'))
with open(ARTIFACTS / 'cross_method_comparison.json') as f:
    cross = json.load(f)
print('Thesis cross-method (for reference):')
print(json.dumps(cross, indent=2))

# Try to load DT importances. They are saved inside dt_rules / not as a JSON;
# re-derive from the fitted DT.
import joblib
dt_obj = joblib.load(ARTIFACTS / 'dt_model.pkl')
feature_cols_thesis = json.load(open(ARTIFACTS / 'feature_cols.json'))
# dt_obj may be a tuple (tree, encoder, ...) or a dict — handle both.
if isinstance(dt_obj, dict):
    dt_tree = dt_obj.get('tree') or dt_obj.get('model')
    dt_feat = dt_obj.get('feature_names', feature_cols_thesis)
elif isinstance(dt_obj, tuple):
    dt_tree = dt_obj[0]
    dt_feat = feature_cols_thesis
else:
    dt_tree = dt_obj
    dt_feat = feature_cols_thesis
dt_importance_arr = dt_tree.feature_importances_
dt_imp = dict(zip(dt_feat, dt_importance_arr.tolist()))
print(f'Loaded DT importances: {len(dt_imp)} features')


In [ ]:
# Map thesis feature names (dot-notation) to HDS-style names so they align with XGBoost cols.
thesis_to_hds = {f: rename_to_hds(f) for f in feature_cols_thesis}

def to_hds_dict(d):
    return {thesis_to_hds.get(k, k): v for k, v in d.items()}

shap_h = to_hds_dict(shap_imp)
lime_h = to_hds_dict(lime_imp)
dt_h = to_hds_dict(dt_imp)

shared = sorted(set(shap_h) & set(lime_h) & set(dt_h) & set(xgb_importance))
print(f'Features shared across all four methods: {len(shared)}')
missing = sorted(set(shap_h) - set(xgb_importance))
if missing:
    print(f'Thesis features NOT in XGBoost ranking: {missing}')


In [ ]:
def rank_series(d, keys):
    s = pd.Series({k: d.get(k, 0.0) for k in keys})
    return s.rank(ascending=False, method='min')

ranks = pd.DataFrame({
    'XGBoost': rank_series(xgb_importance, shared),
    'SHAP': rank_series(shap_h, shared),
    'LIME': rank_series(lime_h, shared),
    'DT': rank_series(dt_h, shared),
})

methods = ['XGBoost', 'SHAP', 'LIME', 'DT']
summary_rows = []
for i, a in enumerate(methods):
    for b in methods[i + 1:]:
        rho = stability_spearman(ranks[a], ranks[b])
        j5 = jaccard_at_k(ranks[a], ranks[b], 5)
        j10 = jaccard_at_k(ranks[a], ranks[b], 10)
        summary_rows.append({'A': a, 'B': b, 'Spearman ρ': round(rho, 4),
                             'Jaccard@5': round(j5, 4), 'Jaccard@10': round(j10, 4)})
summary = pd.DataFrame(summary_rows)
summary


In [ ]:
# Save the comparison + visualise top-15 per method side-by-side.
summary.to_csv(OUT_DIR / 'vendor_vs_thesis_comparison.csv', index=False)
print(f'Wrote {OUT_DIR / "vendor_vs_thesis_comparison.csv"}')

import plotly.graph_objects as go
from plotly.subplots import make_subplots

CATEGORY_COLORS = {
    'Context':    '#FFA500',
    'IH':         '#2CA02C',
    'Customer':   '#1F77B4',
    'Param':      '#17BECF',
    'Booking':    '#9467BD',
    'Precompute': '#008080',
    'Internal':   '#7F7F7F',
}

def short_name(f: str) -> str:
    base = f.split('_', 1)[1] if '_' in f else f
    base = base.split('__')[0]
    parts = base.split('_')
    if len(parts) > 3:
        base = '…' + '_'.join(parts[-3:])
    return base

def top_k_records(d, k, method):
    s = pd.Series(d).sort_values(ascending=False).head(k)
    return pd.DataFrame({
        'Feature':    s.index,
        'Short':      [short_name(f) for f in s.index],
        'Category':   [category_of(f) for f in s.index],
        'Importance': s.values,
        'Method':     method,
    })

panels = [
    top_k_records(xgb_importance, 15, 'XGBoost (vendor)'),
    top_k_records(shap_h,         15, 'SHAP (thesis)'),
    top_k_records(lime_h,         15, 'LIME (thesis)'),
    top_k_records(dt_h,           15, 'DT (thesis)'),
]

fig = make_subplots(
    rows=2, cols=2,
    horizontal_spacing=0.32, vertical_spacing=0.10,
    subplot_titles=[p['Method'].iloc[0] for p in panels],
)
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for panel, (r, c) in zip(panels, positions):
    seen = {}
    labels = []
    for s in panel['Short']:
        seen[s] = seen.get(s, 0) + 1
        labels.append(s if seen[s] == 1 else f"{s} ({seen[s]})")
    panel = panel.assign(Label=labels)
    panel_rev = panel.iloc[::-1].reset_index(drop=True)

    fig.add_trace(
        go.Bar(
            x=panel_rev['Importance'],
            y=panel_rev['Label'],
            orientation='h',
            marker_color=[CATEGORY_COLORS.get(cat, '#888') for cat in panel_rev['Category']],
            hovertext=panel_rev['Feature'],
            hoverinfo='text+x',
            showlegend=False,
        ),
        row=r, col=c,
    )
    fig.update_yaxes(
        categoryorder='array',
        categoryarray=panel_rev['Label'].tolist(),
        row=r, col=c,
        automargin=True,
        tickfont=dict(size=10),
    )
    fig.update_xaxes(title_text='Importance', row=r, col=c)

for cat, color in CATEGORY_COLORS.items():
    fig.add_trace(
        go.Bar(x=[None], y=[None], name=cat,
               marker_color=color, showlegend=True, hoverinfo='skip'),
        row=1, col=1,
    )

fig.update_layout(
    title='Top-15 features per method (L5B15) — colored by category',
    height=1000, width=1200, template='plotly_white',
    legend=dict(title='Category', orientation='h', y=-0.08),
    margin=dict(l=10, r=10, t=80, b=80),
    barmode='group',
)
fig.show()


## 7. Candidate new features (vendor framing)

Pega's notebook frames the question as *"which features outside the current ADM predictor set look promising for inclusion?"*. In our case the XGBoost saw all parquet fields, while Pega's L5B15 model uses only the 23 in `feature_cols.json`. So any XGBoost top-feature that is **NOT** already in the active predictor set is a candidate addition.

In [ ]:
active = set(thesis_to_hds.values())
candidates = (
    full_importance.filter(~pl.col('Feature').is_in(list(active)))
    .head(20)
)
candidates


In [ ]:
candidates.write_csv(OUT_DIR / 'candidate_new_features.csv')
print(f'Wrote {OUT_DIR / "candidate_new_features.csv"}')